In [0]:
from pyspark.sql.functions import col, when, current_timestamp

In [0]:
input_file = 'incremental_sales.bronze.sales'
checkpoint_silver = '/Volumes/incremental_sales/default/metadata/checkpoints/silver'


In [0]:
df = spark.readStream.table(input_file)

In [0]:
# rename columns
cols = {
    "InvoiceNo": "invoice_no",
    "StockCode": "stock_code",
    "Description": "description",
    "Quantity": "quantity",
    "InvoiceDate": "invoice_date",
    "UnitPrice": "unit_price",
    "CustomerID": "customer_id",
    "Country": "country"
}

for col1, col2 in cols.items():
  df = df.withColumnRenamed(col1, col2)

In [0]:
# create error_reason column to validate data and quarantine bad data
df = (
    df.withColumn("error_reason", 
                  when(col("customer_id").isNull(), "missing customer_id")
                  .when(col("quantity")<1, "invalid quantity")
                  .when(col("unit_price")<=0, "invalid price")
                  .when(col("invoice_no").startswith("C"),"cancelled order")
                  )
)

silver_df =(df
    .filter(col("error_reason").isNull())
    .drop("error_reason")
    .withColumn("processed_time", current_timestamp())
)

quarantine_df = (df
                 .filter(col("error_reason").isNotNull())
                 .withColumn("processed_time", current_timestamp())
)

#write good data to silver sales table
(silver_df
    .writeStream
    .format("delta")
    .option("checkpointLocation", f"{checkpoint_silver}/sales")
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable("incremental_sales.silver.sales")
)

#writebad data to silver quarantine table
(quarantine_df
    .writeStream
    .format("delta")
    .option("checkpointLocation", f"{checkpoint_silver}/quarantine")
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable("incremental_sales.silver.quarantine_sales")
)

In [0]:
%sql
select * from incremental_sales.silver.sales limit 5

invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,ingestion_time,source_file,processed_time
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01T08:26:00.000Z,2.55,17850.0,United Kingdom,2026-09-03T11:36:26.445Z,/Volumes/incremental_sales/default/data/landing/sales_2010_12.csv,2026-09-03T11:41:28.284Z
536365,71053,WHITE METAL LANTERN,6,2010-12-01T08:26:00.000Z,3.39,17850.0,United Kingdom,2026-09-03T11:36:26.445Z,/Volumes/incremental_sales/default/data/landing/sales_2010_12.csv,2026-09-03T11:41:28.284Z
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01T08:26:00.000Z,2.75,17850.0,United Kingdom,2026-09-03T11:36:26.445Z,/Volumes/incremental_sales/default/data/landing/sales_2010_12.csv,2026-09-03T11:41:28.284Z
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01T08:26:00.000Z,3.39,17850.0,United Kingdom,2026-09-03T11:36:26.445Z,/Volumes/incremental_sales/default/data/landing/sales_2010_12.csv,2026-09-03T11:41:28.284Z
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01T08:26:00.000Z,3.39,17850.0,United Kingdom,2026-09-03T11:36:26.445Z,/Volumes/incremental_sales/default/data/landing/sales_2010_12.csv,2026-09-03T11:41:28.284Z


In [0]:
%sql
select * from incremental_sales.silver.quarantine_sales limit 5

invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,ingestion_time,source_file,error_reason,processed_time
C573750,85159B,"WHITE TEA,COFFEE,SUGAR JARS",-2,2011-11-01T09:29:00.000Z,1.95,16150.0,United Kingdom,2026-08-28T05:51:05.641Z,/Volumes/workspace/projects/incremental_pipeline/landing/sales_2011_11.csv,invalid quantity,2026-08-28T05:51:33.896Z
C573750,23393,HOME SWEET HOME CUSHION COVER,-1,2011-11-01T09:29:00.000Z,3.75,16150.0,United Kingdom,2026-08-28T05:51:05.641Z,/Volumes/workspace/projects/incremental_pipeline/landing/sales_2011_11.csv,invalid quantity,2026-08-28T05:51:33.896Z
C573750,22109,FULL ENGLISH BREAKFAST PLATE,-2,2011-11-01T09:29:00.000Z,3.75,16150.0,United Kingdom,2026-08-28T05:51:05.641Z,/Volumes/workspace/projects/incremental_pipeline/landing/sales_2011_11.csv,invalid quantity,2026-08-28T05:51:33.896Z
573751,46000U,check,10,2011-11-01T09:33:00.000Z,0,null,United Kingdom,2026-08-28T05:51:05.641Z,/Volumes/workspace/projects/incremental_pipeline/landing/sales_2011_11.csv,missing customer_id,2026-08-28T05:51:33.896Z
573752,46000S,dotcom sales,-90,2011-11-01T09:34:00.000Z,0,null,United Kingdom,2026-08-28T05:51:05.641Z,/Volumes/workspace/projects/incremental_pipeline/landing/sales_2011_11.csv,missing customer_id,2026-08-28T05:51:33.896Z
